In [1]:
import os

import duckdb
import pyarrow.parquet as pq

In [2]:
base_path = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line"
output_dir = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\filtered"

measurements_file = os.path.join(base_path, "measurements_single_line.parquet")
filtered_output_file = os.path.join(output_dir, "filtered_measurements.parquet")
final_output_file = os.path.join(output_dir, "filtered_measurements_encoded.parquet")

## Step 1: Filter columns with PyArrow

In [ ]:
# Columns you want to keep
keep_columns = [
    "measure_step_number", "measure_value", "created_at", "booking_id",
    "book_state", "serial_number_id",
    "station_id", "measurement_name", "measurement_unit",
    "lower_limit", "upper_limit"
]

In [ ]:
# Read the Parquet file into a PyArrow Table
table = pq.read_table(measurements_file)

In [ ]:
# Drop columns not in keep list
columns_to_drop = [col for col in table.column_names if col not in keep_columns]
filtered_table = table.drop(columns_to_drop)

In [ ]:
# Write filtered table back to Parquet
pq.write_table(filtered_table, final_output_file)
print(f"Reduced Parquet saved to: {final_output_file}")

## Step 2: Encode the measurement_name and measurement_unit columns, Merge upper and lower limit

In [3]:
con = duckdb.connect()

In [4]:
# Create encoded dataset with DuckDB (frequency encoding + computed column + drop limits)
con.execute(f"""
COPY (
    WITH base AS (
        SELECT *,
            -- CAST numeric columns early
            TRY_CAST(measure_value AS DOUBLE) AS measure_value_num,
            TRY_CAST(lower_limit AS DOUBLE) AS lower_limit_num,
            TRY_CAST(upper_limit AS DOUBLE) AS upper_limit_num,
            COALESCE(measurement_unit, 'missing') AS measurement_unit_filled,
            -- Frequency encoding for measurement_name
            (SELECT COUNT(*) FROM '{filtered_output_file}' AS sub WHERE sub.measurement_name = main.measurement_name) AS measurement_name_encoded
        FROM '{filtered_output_file}' AS main
    ),
    filtered AS (
        SELECT * FROM base
        WHERE
            measure_value_num IS NOT NULL
            AND lower_limit_num IS NOT NULL
            AND upper_limit_num IS NOT NULL
    ),
    encoded AS (
        SELECT
            measure_step_number,
            measure_value_num AS measure_value,
            created_at,
            booking_id,
            book_state,
            serial_number_id,
            station_id,
            measurement_name_encoded,
            -- Frequency encoding for measurement_unit
            (SELECT COUNT(*) FROM filtered AS sub WHERE sub.measurement_unit_filled = filtered.measurement_unit_filled) AS measurement_unit_encoded,
            -- is_within_limits calculation (all DOUBLEs now)
            (measure_value_num >= lower_limit_num AND measure_value_num <= upper_limit_num)::INTEGER AS is_within_limits
        FROM filtered
    )
    SELECT * FROM encoded
) TO '{final_output_file}' (FORMAT PARQUET, COMPRESSION 'zstd');
""")

print(f"✅ Final encoded Parquet saved to: {final_output_file}")

con.close()

✅ Final encoded Parquet saved to: M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\filtered\filtered_measurements_encoded.parquet
